# Vector Quantized VAE（VQ-VAE）

---
## 目的
VQ-VAE [1]を構築し，連続的な潜在変数ではなく，有限個のベクトルの集合（コードブック）から選択される離散的な潜在変数を用いて画像を再構成する仕組みを理解する．`vae.ipynb`のVAEとの違い（連続な潜在表現 vs 離散な潜在表現）に注目する．

[1] Aaron van den Oord, Oriol Vinyals, Koray Kavukcuoglu, "Neural Discrete Representation Learning," NeurIPS, 2017.

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## VQ-VAEとは
`vae.ipynb`のVAEは，Encoderの出力を正規分布のパラメータ（平均・分散）として扱い，そこから連続的な潜在変数をサンプリングしました．一方，画像・音声・言語といった多くのデータは，背後に離散的な構造（例えば「どの数字が書かれているか」）を持つと考えられます．

VQ-VAE (Vector Quantized VAE) は，Encoderが出力する連続的な特徴ベクトルを，学習可能な有限個のベクトル$\{e_1, \dots, e_K\}$（**コードブック**）の中から最も近いベクトルに置き換える（**量子化**）ことで，離散的な潜在表現を獲得する手法です．

## ネットワークの構造
VQ-VAEのEncoderは，`vae.ipynb`のように画像全体を1つのベクトルへ圧縮するのではなく，畳み込み層を用いて画像を空間的な特徴マップ$z_e(x) \in \mathbb{R}^{D\times H'\times W'}$へ変換します．この特徴マップの**各空間位置**（$H'\times W'$個）のD次元ベクトルを，それぞれ独立にコードブックの中から最も近いベクトルへ置き換えることで，量子化された特徴マップ$z_q(x)$を得ます．Decoderは，この$z_q(x)$から画像を再構成します．

つまり，1枚の画像は「$H'\times W'$個のコード番号（$0 \sim K-1$の離散値）の並び」として離散的に表現されることになります．

## コードブックによる量子化とStraight-Through Estimator
特徴マップの各位置のベクトル$z_e$を，コードブックの中で最もユークリッド距離が近いベクトル$e_k$に置き換えます．

$$
z_q = e_{k^*}, \quad k^* = \operatorname*{argmin}_{k} \|z_e - e_k\|^2
$$

このargmin（最も近いコードを選ぶ操作）は微分できないため，このままでは誤差逆伝播でEncoderを学習できません．そこでVQ-VAEでは，**Straight-Through Estimator**という手法を用います．forward計算では量子化後の$z_q$をそのままDecoderへ渡す一方，backward計算（勾配計算）では，Decoderから伝わってきた勾配を量子化前の$z_e$へそのままコピーして流します．PyTorchでは，`detach()`を使うことで以下のようにこれを実現できます．

```python
z_q = z_e + (z_q - z_e).detach()
```

`(z_q - z_e)`の値はforward計算では通常通り使われますが，`detach()`により，backward計算ではこの部分の勾配が0として扱われます．そのため，上式の右辺全体は，forward計算では`z_q`と同じ値を返しつつ，backward計算では`z_e`と全く同じ勾配を返す，という挙動になります．

また，コードブック$\{e_k\}$とEncoderの出力$z_e$を学習するために，以下の2つの誤差を追加で使用します．

$$
\mathcal{L}_{vq} = \underbrace{\|\mathrm{sg}[z_e] - e_{k^*}\|^2}_{\text{コードブック損失}} + \beta\underbrace{\|z_e - \mathrm{sg}[e_{k^*}]\|^2}_{\text{コミットメント損失}}
$$

$\mathrm{sg}[\cdot]$はstop-gradient（勾配を計算しない）操作を表します．コードブック損失はコードブックのベクトルをEncoderの出力へ近づけ，コミットメント損失はEncoderの出力をコードブックのベクトルへ近づける役割を持ちます．$\beta$（コミットメント損失の重み）は，原論文にならい`0.25`とします．

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)
        self.commitment_cost = commitment_cost

    def forward(self, z_e):
        # z_e: (B, D, H, W) -> (B*H*W, D)
        b, d, h, w = z_e.shape
        flat_z_e = z_e.permute(0, 2, 3, 1).reshape(-1, d)

        # 各ベクトルとコードブックの全エントリとのユークリッド距離を計算し，最も近いコードを選択
        distances = (flat_z_e.pow(2).sum(1, keepdim=True)
                     - 2 * flat_z_e @ self.embedding.weight.t()
                     + self.embedding.weight.pow(2).sum(1))
        indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(indices).view(b, h, w, d).permute(0, 3, 1, 2)

        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss

        # Straight-Through Estimator
        z_q = z_e + (z_q - z_e).detach()

        # コードブックの利用状況（Perplexity）を計算
        encodings = F.one_hot(indices, self.num_embeddings).float()
        avg_probs = encodings.mean(0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        return z_q, vq_loss, perplexity, indices.view(b, h, w)

## ネットワークの作成
Encoder・Decoderは，畳み込み層のみで構成します．入力画像を$32\times32$にリサイズして使用するため，Encoderで2回ストライド2の畳み込みを適用すると，特徴マップのサイズは$8\times8$になります．コードブックのベクトルの次元数（`embedding_dim`）は`64`，コードブックのサイズ（`num_embeddings`，コードの種類数$K$）は`128`とします．

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, embedding_dim=64, num_embeddings=128, commitment_cost=0.25):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 16 -> 8
            nn.Conv2d(64, embedding_dim, kernel_size=3, stride=1, padding=1),  # 8 -> 8
        )
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 8 -> 16
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 16 -> 32
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),  # Sigmoidは適用しない（誤差関数側でまとめて適用する）
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, perplexity, indices = self.vq(z_e)
        x_hat = self.decoder(z_q)
        return x_hat, vq_loss, perplexity, indices

## データセット，ネットワーク，最適化関数の設定
データセットにはMNISTを使用し，`dcgan.ipynb`と同様に$32\times32$へリサイズします．最適化手法にはAdam optimizer（学習率$2\times 10^{-4}$）を使用します．

In [ ]:
batch_size = 128

transform = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])
mnist_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(mnist_data, batch_size=batch_size, shuffle=True)

model = VQVAE(embedding_dim=64, num_embeddings=128, commitment_cost=0.25).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

## 学習
誤差関数は，再構成誤差（Binary Cross Entropy）と，`vq`モジュールが返す`vq_loss`（コードブック損失＋コミットメント損失）の和です．あわせて，コードブックの利用状況を表す**Perplexity**（$=\exp(\text{エントロピー})$）も表示します．Perplexityは，コードが1種類しか使われていない場合は`1`に，`K`種類のコードが均等に使われている場合は`K`に近づきます．学習エポック数を`20`として学習します．

In [ ]:
epoch_num = 20

model.train()
start = time.time()
for epoch in range(1, epoch_num + 1):
    sum_loss, sum_recon, sum_vq, sum_ppl = 0.0, 0.0, 0.0, 0.0
    for x, _ in train_loader:
        x = x.to(device)

        x_hat, vq_loss, perplexity, _ = model(x)
        recon_loss = F.binary_cross_entropy_with_logits(x_hat, x, reduction='mean')
        loss = recon_loss + vq_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_recon += recon_loss.item()
        sum_vq += vq_loss.item()
        sum_ppl += perplexity.item()

    n = len(train_loader)
    print(f'epoch: {epoch}, loss: {sum_loss/n:.4f}, recon: {sum_recon/n:.4f}, vq_loss: {sum_vq/n:.4f}, perplexity: {sum_ppl/n:.2f}, elapsed_time: {time.time()-start:.2f}')

## 学習済みモデルを用いた画像の復元
評価用データからランダムに画像をサンプルし，VQ-VAEによる再構成結果を確認します．`vae.ipynb`と同様，Decoderの出力は生のロジットであるため，画像として表示する前に`torch.sigmoid`を適用します．

In [ ]:
mnist_testdata = datasets.MNIST(root='./data', train=False, transform=transform)
test_loader = DataLoader(mnist_testdata, batch_size=10, shuffle=True)

model.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x.to(device)
    x_hat, vq_loss, perplexity, indices = model(x)
    x_hat = torch.sigmoid(x_hat)

x_cpu = x.cpu()
x_hat_cpu = x_hat.cpu()

fig, axes = plt.subplots(2, 10, figsize=(14, 2.8))
for i in range(10):
    axes[0, i].imshow(x_cpu[i, 0], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_hat_cpu[i, 0], cmap='gray'); axes[1, i].axis('off')
fig.suptitle('input (top) / reconstruction (bottom)')
plt.show()

## 離散潜在表現（コード割り当て）の可視化
1枚の画像が，実際にどのようなコード番号の並びとして離散的に表現されているかを可視化します．各画像の下に示す$8\times8$のマス目は，対応する空間位置で選択されたコードブックのインデックス（`0`〜`127`）を色分けして表示したものです．同じ色は同じコードが選択されていることを表します．

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    axes[0, i].imshow(x_cpu[i, 0], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(indices[i].cpu(), cmap='tab20'); axes[1, i].axis('off')
fig.suptitle('input (top) / assigned code indices (bottom)')
plt.show()

## 課題

1. コードブックのサイズ（`num_embeddings`）や次元数（`embedding_dim`）を変更して学習し，再構成の質やPerplexity（コードブックの利用状況）がどのように変化するか確認してください．
2. コミットメント損失の重み`commitment_cost`を変更して学習し，学習の安定性や再構成の質にどのような影響があるか確認してください．
3. `sq_vae.ipynb`のSQ-VAEと比較し，再構成の質やコードブックの利用状況（Perplexity）にどのような違いがあるか確認してください．